<a href="https://colab.research.google.com/github/Rahul19873/ML-and-Deep-learning-project/blob/main/real_and_fake_news.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text  import TfidfVectorizer

from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score,confusion_matrix
from sklearn.metrics import classification_report





In [ ]:
fake=pd.read_csv('Fake.csv', on_bad_lines='skip', engine='python')
true=pd.read_csv('True.csv', on_bad_lines='skip', engine='python')

In [ ]:
# create label
fake['label']=0
true['label']=1

In [ ]:
# combine the dataset

df=pd.concat([fake,true],ignore_index=True)

#shuffle dataset
df=df.sample(frac=1,random_state=42)

In [ ]:
# check data set
df.head()
df.info()
df.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
Index: 7534 entries, 3649 to 7270
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    7534 non-null   object
 1   text     7534 non-null   object
 2   subject  7534 non-null   object
 3   date     7534 non-null   object
 4   label    7534 non-null   int64 
dtypes: int64(1), object(4)
memory usage: 353.2+ KB


,0
title,0
text,0
subject,0
date,0
label,0


In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Ensure stopwords are downloaded
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')

ps=PorterStemmer()
def clean_text(text):
  text = re.sub('[^a-zA-Z]', ' ', text)
  text=text.lower()
  words= text.split()

  words = [
        ps.stem(word)
        for word in words
        if word not in stopwords.words('english')
  ]

  return " ".join(words)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [18]:
# apply cleaning
df['content']=df['title'] + ' ' +df['text']

df["content"]=df["content"].apply(clean_text)

In [21]:
# features and labels
X=df['content']
y=df['label']

In [22]:
# converting word into numerics using Tf-idf
tfidf=TfidfVectorizer(max_features=10000)
X=tfidf.fit_transform(X)





In [25]:
# train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)



In [26]:
# train the model
model= PassiveAggressiveClassifier(max_iter=100)
model.fit(X_train,y_train)

PassiveAggressiveClassifier(max_iter=100)

In [34]:
# Predection the model
model.predict(X_test)

array([1, 0, 0, ..., 0, 0, 1])

In [36]:
# accuracy
accuracy_score(y_test,model.predict(X_test))

0.9986728599867286

In [37]:
print(confusion_matrix(y_test,model.predict(X_test)))

[[874   1]
 [  1 631]]


In [39]:
#classification report
print(classification_report(y_test,model.predict(X_test)))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00       875
           1       1.00      1.00      1.00       632

    accuracy                           1.00      1507
   macro avg       1.00      1.00      1.00      1507
weighted avg       1.00      1.00      1.00      1507



In [42]:
# test the custom news
news=[ 'government anounces new education policy for school'
]
news=tfidf.transform(news)
result=model.predict(news)
print(result)


[1]


In [43]:
# save the model
import pickle
pickle.dump(model,open('model.pickle','wb'))
pickle.dump(tfidf,open('tfidf.pickle','wb'))


In [46]:
# build a simple predection script
user_input=input('enter the news:')
clean= clean_text(user_input)
vector=tfidf.transform([clean])
predection = model.predict(vector)
if predection[0]==1:
  print('Real News')
else:
  print('Fake news')

enter the news:it is bad 
Fake news
